**SETUP CONNECTION**

In [16]:
import psycopg2
from db_secrets import user,password
import pandas as pd 


DATABASE = "recitation_3"



conection = psycopg2.connect(database=DATABASE,user=user,password=password, host="localhost")

cursor = conection.cursor()

def print_query(query):
    try:
        cursor.execute(query)
        labels = [col.name for col in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=labels)
    except Exception as e:
        conection.rollback()
        print("Error:", e)
def print_query_analyze_explain(query):
    try:
        cursor.execute("EXPLAIN (ANALYZE, BUFFERS)"+query)
        
        for row in cursor.fetchall():
            print(row[0])

    except Exception as e:
        conection.rollback()
        print("Error:", e)


        


query = """
SELECT DISTINCT con FROM star 
    ORDER BY con DESC;
"""

print_query(query)





,con
0,NaN
1,Vul
2,Vol
3,Vir
4,Vel
...,...
84,Aqr
85,Aql
86,Aps
87,Ant


In [17]:
print_query_analyze_explain(query)


Sort  (cost=4459.05..4459.27 rows=88 width=4) (actual time=122.568..122.581 rows=89 loops=1)
  Sort Key: con DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=2960
  ->  HashAggregate  (cost=4455.32..4456.20 rows=88 width=4) (actual time=122.305..122.336 rows=89 loops=1)
        Group Key: con
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=2960
        ->  Seq Scan on star  (cost=0.00..4156.26 rows=119626 width=4) (actual time=0.025..26.746 rows=119626 loops=1)
              Buffers: shared hit=2960
Planning Time: 0.142 ms
Execution Time: 122.674 ms


### 9. Stars with sp ectral typ e `M' in constellation Pisces.

In [18]:
query = """ 
SELECT s.id, s.proper_name, s.spectral_type
FROM star AS s
JOIN constellation AS c ON s.con = c.abbr
WHERE c.name = 'Pisces'
  AND s.spectral_type LIKE 'M%'; -- this makes that it checks the first letter
"""

print_query(query)


,id,proper_name,spectral_type
0,6,None,M0V:
1,154,None,M3III
2,263,None,M1
3,578,None,M...
4,895,None,M2
...,...,...,...
110,118087,None,M3
111,118103,None,M1.5
112,119571,None,M5
113,119577,None,M3.5:


In [19]:
print_query_analyze_explain(query)

Hash Join  (cost=2.11..4478.73 rows=63 width=16) (actual time=0.155..59.166 rows=115 loops=1)
  Hash Cond: ((s.con)::text = (c.abbr)::text)
  Buffers: shared hit=2961
  ->  Seq Scan on star s  (cost=0.00..4455.32 rows=5510 width=20) (actual time=0.030..56.561 rows=5243 loops=1)
        Filter: (spectral_type ~~ 'M%'::text)
        Rows Removed by Filter: 114383
        Buffers: shared hit=2960
  ->  Hash  (cost=2.10..2.10 rows=1 width=4) (actual time=0.093..0.096 rows=1 loops=1)
        Buckets: 1024  Batches: 1  Memory Usage: 9kB
        Buffers: shared hit=1
        ->  Seq Scan on constellation c  (cost=0.00..2.10 rows=1 width=4) (actual time=0.043..0.052 rows=1 loops=1)
              Filter: (name = 'Pisces'::text)
              Rows Removed by Filter: 87
              Buffers: shared hit=1
Planning Time: 0.545 ms
Execution Time: 59.315 ms


### 10. The 100 stars closest to the sun.

In [20]:
query = """ 
SELECT n.id,
       n.proper_name,
       ABS(n.x_pc - s.x_pc) AS x_diff,
       ABS(n.y_pc - s.y_pc) AS y_diff,
       ABS(n.z_pc - s.z_pc) AS z_diff,
       (ABS(n.x_pc - s.x_pc) * ABS(n.x_pc - s.x_pc) + 
        ABS(n.y_pc - s.y_pc) * ABS(n.y_pc - s.y_pc) +
        ABS(n.z_pc - s.z_pc) * ABS(n.z_pc - s.z_pc)) AS distance_pow_2
    FROM star as n CROSS JOIN star as s 
    WHERE 
        s.proper_name ILIKE 'Sol' AND 
        n.id <> s.id
    ORDER BY distance_pow_2 ASC LIMIT 100; 

"""

print_query(query)

,id,proper_name,x_diff,y_diff,z_diff,distance_pow_2
0,70666,Proxima Centauri,0.472269,0.361451,1.151219,1.678990
1,71453,Toliman,0.495186,0.413973,1.156674,1.754478
2,71456,Rigil Kentaurus,0.495208,0.414084,1.156625,1.754478
3,87665,Barnard's Star,0.017378,1.816613,0.149123,3.322622
4,118720,Wolf 359,2.282816,0.649334,0.292057,5.718181
...,...,...,...,...,...,...
95,75841,NaN,2.678399,3.561312,3.911408,35.155877
96,84147,Guniibuu,1.027051,5.204798,2.656955,35.204166
97,3814,Achird,3.098457,0.674227,5.038376,35.440251
98,118021,NaN,3.098391,0.673787,5.038550,35.441002


In [21]:
print_query_analyze_explain(query)

Limit  (cost=17226.77..17238.27 rows=100 width=44) (actual time=205.247..205.501 rows=100 loops=1)
  Buffers: shared hit=5957
  ->  Gather Merge  (cost=17226.77..25319.09 rows=70368 width=44) (actual time=205.244..205.483 rows=100 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=5957
        ->  Sort  (cost=16226.76..16402.68 rows=70368 width=44) (actual time=118.013..118.042 rows=50 loops=2)
              Sort Key: ((((abs((n.x_pc - s.x_pc)) * abs((n.x_pc - s.x_pc))) + (abs((n.y_pc - s.y_pc)) * abs((n.y_pc - s.y_pc)))) + (abs((n.z_pc - s.z_pc)) * abs((n.z_pc - s.z_pc)))))
              Sort Method: top-N heapsort  Memory: 37kB
              Buffers: shared hit=5957
              Worker 0:  Sort Method: quicksort  Memory: 25kB
              ->  Nested Loop  (cost=0.00..13537.35 rows=70368 width=44) (actual time=15.764..86.037 rows=59812 loops=2)
                    Join Filter: (n.id <> s.id)
                    Rows Removed by Join Filter: 0


###  11. The 100 stars closest to the star with proper name Antares (do not plug in the coordinates of this star
in SQL. The literal `Antares' must be in your SQL)

In [22]:
query = """ 
SELECT n.id,
       n.proper_name,
       ABS(n.x_pc - s.x_pc) AS x_diff,
       ABS(n.y_pc - s.y_pc) AS y_diff,
       ABS(n.z_pc - s.z_pc) AS z_diff,
       (ABS(n.x_pc - s.x_pc) * ABS(n.x_pc - s.x_pc) + 
        ABS(n.y_pc - s.y_pc) * ABS(n.y_pc - s.y_pc) +
        ABS(n.z_pc - s.z_pc) * ABS(n.z_pc - s.z_pc)) AS distance_pow_2
    FROM star as n CROSS JOIN star as s 
    WHERE 
        s.proper_name ILIKE 'Antares' AND 
        n.id <> s.id
    ORDER BY distance_pow_2 ASC LIMIT 100; 

"""

print_query(query)

,id,proper_name,x_diff,y_diff,z_diff,distance_pow_2
0,80938,NaN,3.078432,2.190751,3.432893,26.060888
1,80944,NaN,4.739820,1.898407,1.749059,29.129050
2,80574,NaN,0.199982,0.999385,5.425223,30.471808
3,79807,NaN,3.978069,5.680998,5.995074,84.039684
4,81279,NaN,7.774334,2.475877,5.523168,97.075621
...,...,...,...,...,...,...
95,81898,NaN,4.941132,22.128246,12.965781,682.185533
96,83402,NaN,22.515609,9.595595,9.436476,688.075171
97,78810,NaN,3.181474,24.066836,10.451689,698.572175
98,78113,NaN,11.106021,19.838511,13.626258,702.585128


In [23]:
print_query_analyze_explain(query)

Limit  (cost=17226.77..17238.27 rows=100 width=44) (actual time=308.493..314.851 rows=100 loops=1)
  Buffers: shared hit=5957
  ->  Gather Merge  (cost=17226.77..25319.09 rows=70368 width=44) (actual time=308.490..314.830 rows=100 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=5957
        ->  Sort  (cost=16226.76..16402.68 rows=70368 width=44) (actual time=166.049..166.058 rows=50 loops=2)
              Sort Key: ((((abs((n.x_pc - s.x_pc)) * abs((n.x_pc - s.x_pc))) + (abs((n.y_pc - s.y_pc)) * abs((n.y_pc - s.y_pc)))) + (abs((n.z_pc - s.z_pc)) * abs((n.z_pc - s.z_pc)))))
              Sort Method: quicksort  Memory: 25kB
              Buffers: shared hit=5957
              Worker 0:  Sort Method: top-N heapsort  Memory: 36kB
              ->  Nested Loop  (cost=0.00..13537.35 rows=70368 width=44) (actual time=22.903..119.822 rows=59812 loops=2)
                    Join Filter: (n.id <> s.id)
                    Rows Removed by Join Filter: 0

### 12. The 100 closest exoplanets to the sun. Measure this based on the distance of the exoplanets star.

In [24]:
query = """ 
SELECT e.name, 
        n.id,
        e.num_planets,
        (ABS(n.x_pc - s.x_pc) * ABS(n.x_pc - s.x_pc) + 
        ABS(n.y_pc - s.y_pc) * ABS(n.y_pc - s.y_pc) +
        ABS(n.z_pc - s.z_pc) * ABS(n.z_pc - s.z_pc)) AS distance_pow_2
    FROM star as n CROSS JOIN star as s 
    JOIN exoplanet e
    ON e.star_id = n.id
    WHERE 
        s.proper_name ILIKE 'Sol' AND 
        n.id <> s.id
    ORDER BY distance_pow_2 ASC LIMIT 100; 

"""
print_query(query)

,name,id,num_planets,distance_pow_2
0,Proxima Cen b,70666,2,1.678990
1,Proxima Cen d,70666,2,1.678990
2,Barnard d,87665,4,3.322622
3,Barnard c,87665,4,3.322622
4,Barnard b,87665,4,3.322622
...,...,...,...,...
95,GJ 536 b,68250,2,100.562884
96,GJ 436 b,56916,1,102.841722
97,GJ 649 b,82788,1,107.009492
98,HD 62509 b,37718,1,107.297166


In [25]:
print_query_analyze_explain(query)

Limit  (cost=6990.12..6990.37 rows=100 width=28) (actual time=57.948..57.983 rows=100 loops=1)
  Buffers: shared hit=5682
  ->  Sort  (cost=6990.12..6993.16 rows=1217 width=28) (actual time=57.945..57.962 rows=100 loops=1)
        Sort Key: ((((abs((n.x_pc - s.x_pc)) * abs((n.x_pc - s.x_pc))) + (abs((n.y_pc - s.y_pc)) * abs((n.y_pc - s.y_pc)))) + (abs((n.z_pc - s.z_pc)) * abs((n.z_pc - s.z_pc)))))
        Sort Method: top-N heapsort  Memory: 36kB
        Buffers: shared hit=5682
        ->  Nested Loop  (cost=0.30..6943.60 rows=1217 width=28) (actual time=0.561..56.571 rows=1217 loops=1)
              Join Filter: (n.id <> s.id)
              Buffers: shared hit=5682
              ->  Seq Scan on star s  (cost=0.00..4455.32 rows=1 width=28) (actual time=0.030..33.352 rows=1 loops=1)
                    Filter: (proper_name ~~* 'Sol'::text)
                    Rows Removed by Filter: 119625
                    Buffers: shared hit=2960
              ->  Nested Loop  (cost=0.30..2421.34 r

### 13. How many exoplanets per constellation?

In [26]:
query = """ 

SELECT c.name, COUNT(e.name) AS num_planets
FROM constellation AS c JOIN 
star AS s ON s.con = c.abbr
LEFT JOIN exoplanet AS e ON 
s.id = e.star_id
GROUP BY c.name
ORDER BY num_planets ASC;

"""

print_query(query)

,name,num_planets
0,Columba,0
1,Crux,2
2,Caelum,2
3,Lacerta,2
4,Triangulum Australe,2
...,...,...
83,Hydra,37
84,Ursa Major,39
85,Virgo,42
86,Draco,47


In [27]:
print_query_analyze_explain(query)

Sort  (cost=8174.65..8174.87 rows=88 width=16) (actual time=289.803..289.820 rows=88 loops=1)
  Sort Key: (count(e.name))
  Sort Method: quicksort  Memory: 28kB
  Buffers: shared hit=3154
  ->  HashAggregate  (cost=8170.92..8171.80 rows=88 width=16) (actual time=289.716..289.748 rows=88 loops=1)
        Group Key: c.name
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=3154
        ->  Hash Join  (cost=5654.57..7572.79 rows=119626 width=20) (actual time=140.384..237.831 rows=119999 loops=1)
              Hash Cond: ((s.con)::text = (c.abbr)::text)
              Buffers: shared hit=3154
              ->  Hash Right Join  (cost=5651.59..5924.96 rows=119626 width=16) (actual time=140.230..182.055 rows=120000 loops=1)
                    Hash Cond: (e.star_id = s.id)
                    Buffers: shared hit=3153
                    ->  Seq Scan on exoplanet e  (cost=0.00..256.66 rows=6366 width=16) (actual time=0.010..1.110 rows=6366 loops=1)
                          Buff

### 14. There is star that is not placed in its correct constellation. Which star is that

In [28]:
query = """ 
WITH calculated_stars AS (
        SELECT  id, 
                proper_name,
                con,
                constellation_at(ra_deg,dec_deg) as calculated_con
    FROM star
)
SELECT  id, 
        proper_name,
        con,
        calculated_con
FROM calculated_stars
WHERE con <> calculated_con; 
"""

print_query(query)

,id,proper_name,con,calculated_con
0,19173,NaN,Ret,Hyi
1,67599,NaN,Cha,Aps
2,69693,NaN,Cen,Lup
3,93308,NaN,Sge,Vul
4,95468,NaN,Cyg,Dra
5,118234,NaN,Cas,Cam
6,119623,NaN,CMi,Gem
7,15164,NaN,Cas,Cam
8,39974,NaN,Mon,Hya
9,47381,NaN,Hya,Sex


In [29]:
print_query_analyze_explain(query)

Gather  (cost=1000.00..51838.46 rows=119028 width=48) (actual time=37316.842..239775.103 rows=16 loops=1)
  Workers Planned: 1
  Workers Launched: 1
  Buffers: shared hit=721569
  ->  Parallel Seq Scan on star  (cost=0.00..38935.66 rows=70016 width=48) (actual time=33470.540..239714.925 rows=8 loops=2)
        Filter: ((con)::text <> (constellation_at(ra_deg, dec_deg))::text)
        Rows Removed by Filter: 59805
        Buffers: shared hit=721569
Planning Time: 0.386 ms
Execution Time: 239775.161 ms


### 15. Give the stars within a distance of 1.0 pc's of a star with spectral type `M'

In [ ]:
query = """ 
WITH distances AS  (SELECT n.id,
       n.proper_name,
       ABS(n.x_pc - s.x_pc) AS x_diff,
       ABS(n.y_pc - s.y_pc) AS y_diff,
       ABS(n.z_pc - s.z_pc) AS z_diff,
       (ABS(n.x_pc - s.x_pc) * ABS(n.x_pc - s.x_pc) + 
        ABS(n.y_pc - s.y_pc) * ABS(n.y_pc - s.y_pc) +
        ABS(n.z_pc - s.z_pc) * ABS(n.z_pc - s.z_pc)) AS distance_pow_2
    FROM star as n CROSS JOIN star as s 
    WHERE 
        s.spectral_type LIKE 'M%' AND 
        n.id <> s.id
)
SELECT id,
       proper_name,
       MIN(distance_pow_2) AS distance_pow_2
FROM distances
WHERE distance_pow_2 < 1
GROUP BY id, proper_name
ORDER BY distance_pow_2 ASC;
"""
print_query(query)

In [ ]:
print_query_analyze_explain(query)

Sort  (cost=4459.05..4459.27 rows=88 width=4) (actual time=125.049..125.061 rows=89 loops=1)
  Sort Key: con DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=2960
  ->  HashAggregate  (cost=4455.32..4456.20 rows=88 width=4) (actual time=124.883..124.906 rows=89 loops=1)
        Group Key: con
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=2960
        ->  Seq Scan on star  (cost=0.00..4156.26 rows=119626 width=4) (actual time=0.016..26.680 rows=119626 loops=1)
              Buffers: shared hit=2960
Planning Time: 0.153 ms
Execution Time: 125.132 ms


### 16. Give the stars not within a distance of 50 pc of a star with spectral type `M'

In [ ]:
query = """ 
WITH distances AS  (SELECT n.id,
       n.proper_name,
       ABS(n.x_pc - s.x_pc) AS x_diff,
       ABS(n.y_pc - s.y_pc) AS y_diff,
       ABS(n.z_pc - s.z_pc) AS z_diff,
       (ABS(n.x_pc - s.x_pc) * ABS(n.x_pc - s.x_pc) + 
        ABS(n.y_pc - s.y_pc) * ABS(n.y_pc - s.y_pc) +
        ABS(n.z_pc - s.z_pc) * ABS(n.z_pc - s.z_pc)) AS distance_pow_2
    FROM star as n CROSS JOIN star as s 
    WHERE 
        s.spectral_type LIKE 'M%' AND 
        n.id <> s.id
)
SELECT id,
       proper_name,
       MIN(distance_pow_2) AS distance_pow_2
FROM distances
GROUP BY id, proper_name
HAVING MIN(distance_pow_2) >= 2500
ORDER BY distance_pow_2 ASC;

"""

print_query(query)

,id,proper_name,distance_pow_2
0,56446,NaN,2.500066e+03
1,10066,NaN,2.500172e+03
2,25737,NaN,2.500333e+03
3,88566,NaN,2.500533e+03
4,36579,NaN,2.500645e+03
...,...,...,...
23713,92178,NaN,4.877301e+08
23714,74328,NaN,4.924896e+08
23715,4935,NaN,4.947995e+08
23716,90466,NaN,5.046090e+08


In [ ]:
print_query_analyze_explain(query)

Sort  (cost=4459.05..4459.27 rows=88 width=4) (actual time=163.269..163.281 rows=89 loops=1)
  Sort Key: con DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=2960
  ->  HashAggregate  (cost=4455.32..4456.20 rows=88 width=4) (actual time=163.121..163.141 rows=89 loops=1)
        Group Key: con
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=2960
        ->  Seq Scan on star  (cost=0.00..4156.26 rows=119626 width=4) (actual time=0.016..33.788 rows=119626 loops=1)
              Buffers: shared hit=2960
Planning Time: 0.181 ms
Execution Time: 163.384 ms


### 17. This will take some time, but make 1 million new galaxies via:

```
select make_galaxies(1000000);
```


### 18.  Give the dso's in virgo with a redshift greater than .5

In [ ]:
query = """ 
SELECT d.name, d.redshift
FROM dso AS d JOIN constellation AS c 
ON d.con = c.abbr
WHERE c.name ILIKE 'virgo' AND d.redshift > 0.5; 
"""

print_query(query)



,name,redshift
0,7e19235,0.513721
1,af74af7,0.595682
2,0deb795,0.861880
3,6c785cf,0.873075
4,5b57875,0.689096
...,...,...
10075,ca0c8c7,0.624937
10076,d59ead0,0.929465
10077,53568fd,0.580831
10078,9897680,0.714586


In [ ]:
print_query_analyze_explain(query)

Sort  (cost=4459.05..4459.27 rows=88 width=4) (actual time=122.187..122.199 rows=89 loops=1)
  Sort Key: con DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=2960
  ->  HashAggregate  (cost=4455.32..4456.20 rows=88 width=4) (actual time=122.011..122.033 rows=89 loops=1)
        Group Key: con
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=2960
        ->  Seq Scan on star  (cost=0.00..4156.26 rows=119626 width=4) (actual time=0.011..26.253 rows=119626 loops=1)
              Buffers: shared hit=2960
Planning Time: 0.126 ms
Execution Time: 122.259 ms


### 19. Give the exoplanets that seem most likely to support human life.

In [ ]:
query = """ 
SELECT name,
       radius_earth,
       mass_earth,
       insolation_earth, -- how much energy it recievs from form its start 
       eq_temp_k, -- estimated equilibrium temp 
       eccentricity, -- how stretched its orbit is
       (
           POWER(radius_earth - 1, 2) +
           POWER(mass_earth - 1, 2) +
           POWER(insolation_earth - 1, 2) +
           POWER(eq_temp_k - 255, 2) +
           POWER(eccentricity - 0.0167, 2)
       ) AS earth_distance_pow_2
FROM exoplanet
WHERE radius_earth IS NOT NULL
  AND mass_earth IS NOT NULL
  AND insolation_earth IS NOT NULL
  AND eq_temp_k IS NOT NULL
  AND eccentricity IS NOT NULL
ORDER BY earth_distance_pow_2 ASC
LIMIT 100;
"""

print_query(query)

,name,radius_earth,mass_earth,insolation_earth,eq_temp_k,eccentricity,earth_distance_pow_2
0,TOI-2257 b,2.194000,5.450,0.6703,256.00,0.4960,22.566567
1,TRAPPIST-1 e,0.920000,0.692,0.6460,249.70,0.0051,28.316715
2,HD 40307 g,2.560000,7.100,0.6378,255.00,0.2900,39.849482
3,TOI-7166 b,2.010000,4.700,1.0700,249.00,0.0000,50.715279
4,Kepler-991 b,2.540000,6.990,1.3710,260.00,0.0000,63.389620
...,...,...,...,...,...,...,...
95,HD 216520 c,3.030000,9.440,1.2669,295.22,0.1200,1693.084807
96,Kepler-1090 b,2.250000,5.690,1.8250,297.00,0.0000,1788.239504
97,TOI-2095 c,1.330000,7.400,1.8400,297.00,0.0000,1805.774779
98,Kepler-196 d,2.660657,7.560,1.8000,297.00,0.0000,1810.431660


In [ ]:
print_query_analyze_explain(query)

Sort  (cost=4459.05..4459.27 rows=88 width=4) (actual time=124.085..124.096 rows=89 loops=1)
  Sort Key: con DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=2960
  ->  HashAggregate  (cost=4455.32..4456.20 rows=88 width=4) (actual time=123.933..123.954 rows=89 loops=1)
        Group Key: con
        Batches: 1  Memory Usage: 24kB
        Buffers: shared hit=2960
        ->  Seq Scan on star  (cost=0.00..4156.26 rows=119626 width=4) (actual time=0.015..26.612 rows=119626 loops=1)
              Buffers: shared hit=2960
Planning Time: 0.187 ms
Execution Time: 124.190 ms


20. Give an example of the simplest query you can think of that will not return within hours if not days. . .


this command 
```
SELECT COUNT(*)
FROM star AS a
CROSS JOIN star AS b
CROSS JOIN star AS c;
```

with rougly 120 000 stars creates 1 728 000 000 000 000 combinations 